# Shikibu 学習（Colab 本番用・M4）

このノートブックは、**Google Colab の L4 ランタイム**で評価関数の学習を回すためのものです。
学習は裏のプロセスで走るので、表示のセルを止めても、ブラウザを再読み込みしても続きます。
学習が終わったら、重みファイル（`.shk`）を書き出し、量子化の確認をします。

使う前に次を確認してください。
- ランタイムは **GPU（L4）** を選ぶ。
- Google Drive に次を置く。
  - `MyDrive/DainagonProject/匠生用/データセット/psv/PonkotsuWCSC35_unique_*.psv`：学習用の PSV（設定の `train_glob`）
  - `MyDrive/Shikibu/`：リポジトリ（無ければ GitHub から取得する）
  - `MyDrive/shikibu_wheels/`：`shk_feat-*.whl`（無ければビルドして保存する）
  - `MyDrive/shikibu/runs/<設定名>/`：学習の出力（`checkpoint.pt`・`log.jsonl`・`status.json`・`.shk`）が自動で作られる
- 学習は**途中で切れる前提**。30分ごとに `checkpoint.pt` を保存する。ランタイムが切れたら、同じセルをもう一度実行すれば続きから再開する。

## 1. Google Drive をマウントする

Drive にあるリポジトリ・wheel・データ・出力を使うため、最初にマウントします。
初回は認証画面が出るので、表示に従って許可してください。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')

## 2. リポジトリを取得（または更新）する

- `REPO`：リポジトリの場所。Drive に既にあれば `git pull`、無ければ GitHub から `git clone` します。
- `BRANCH`：学習コードのブランチ（M4 の成果が入っているブランチ）。

In [ ]:
import os
import subprocess
import glob
import json

REPO_URL = "https://github.com/Harumichi-Toshiki/Shikibu.git"   # 取得元（Drive に無いとき）
BRANCH = "claude/magical-wright-5hukjw"                          # 学習コードのブランチ（M4）
REPO = "/content/drive/MyDrive/Shikibu"                          # Drive 上のリポジトリ
WHEEL_DIR = "/content/drive/MyDrive/shikibu_wheels"              # wheel の保存先
RUNS_DIR = "/content/drive/MyDrive/shikibu/runs"                 # 学習の出力先

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("リポジトリ:", REPO, "/ ブランチ:", BRANCH)

## 3. wheel（`shikibu_feat`）をインストールする

- Drive に `shk_feat-*.whl` があれば、それを入れるだけで済みます（数秒）。
- 無ければ Rust をインストールし、`maturin build --release` で wheel を作って Drive に保存します（初回のみ）。
- ソースを変えたら `MyDrive/shikibu_wheels` の wheel を消してから実行し直してください。

In [ ]:
# 既存の wheel を探す。複数あれば一番新しいものを使う
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)

if wheels:
    # 保存済みの wheel を入れる（ビルドは不要）
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    # Rust が無ければ入れる（ランタイムごとに一度だけ必要）
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])

    # wheel をビルドして Drive に保存し、そのまま入れる
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml",
                    "-o", WHEEL_DIR], cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして Drive に保存しました:", built[-1])

# 動くことを確認する（PyTorch は Colab に入っている）
import torch
import shikibu_feat
print("torch", torch.__version__, "/ CUDA:", torch.cuda.is_available(), "/ GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("論理コア数:", os.cpu_count(), "/ shikibu_feat OK, v0 の次元数:", shikibu_feat.num_features(0))

## 4. 設定を選ぶ（このセッションの量を決める）

- `CONFIG_NAME`：`colab_v0`（RayPair_v0）か `colab_v1`（RayPair_v1）。設定の中身は `configs/<名前>.toml`。
- `SESSION_MAX_STEPS`：**通算の更新回数**の上限（設定の `max_steps`）。`None` なら設定ファイルの値を使う。
  このセッションで区切りたい更新回数を入れる（例：`60000`）。次のセッションでは、より大きい値にして再実行する。
- 出力は `MyDrive/shikibu/runs/<CONFIG_NAME>/` に保存され、`--resume` で続きから回せます。

In [ ]:
import re
import tomllib

CONFIG_NAME = "colab_v0"   # "colab_v0" または "colab_v1"
SESSION_MAX_STEPS = None   # None なら設定の max_steps。数値なら通算の更新回数の上限

src = os.path.join(REPO, "configs", CONFIG_NAME + ".toml")
text = open(src, encoding="utf-8").read()
if SESSION_MAX_STEPS is not None:
    text, n = re.subn(r"^max_steps = \d+", "max_steps = %d" % SESSION_MAX_STEPS, text, flags=re.M)
    assert n == 1, "max_steps の行が見つからない"

CONFIG = "/content/run_config.toml"   # このセッション用の設定（リポジトリは書き換えない）
with open(CONFIG, "w", encoding="utf-8") as f:
    f.write(text)
cfg = tomllib.loads(text)
OUT_DIR = cfg["output"]["out_dir"]
os.makedirs(OUT_DIR, exist_ok=True)

# データの確認：学習用の PSV が見えるか
shards = sorted(glob.glob(cfg["data"]["train_glob"]))
print("学習用の PSV:", len(shards), "個")
for p in shards:
    print("  ", os.path.basename(p), "%.2f GB" % (os.path.getsize(p) / 1e9))
print("出力先:", OUT_DIR, "/ checkpoint:", "あり（--resume で続きから）" if os.path.exists(os.path.join(OUT_DIR, "checkpoint.pt")) else "なし（最初から）")
print("バッチ:", cfg["train"]["batch_size"], "/ 総更新回数:", cfg["train"]["max_steps"], "/ シャッフルバッファ:", cfg["data"]["shuffle_buffer"])

## 5. 学習を裏で開始する（`mon.start`）

- `checkpoint.pt` があれば `--resume` を付けて続きから学習します。無ければ最初から学習します。
- 学習は裏のプロセスで動くので、このセルは一瞬で終わります。
- 止めるときは `mon.stop()` を実行します（最後の保存より後の分は失われます。次の `mon.start` で最後の保存から再開します）。
- 設定の `max_steps` に達すると、保存して終わります（この終了はエラーではありません）。

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO, "tools"))
import shk_monitor as mon

mon.start(CONFIG, repo=REPO)

## 6. 進み具合を表示する（`mon.watch`）

15秒ごとに、段階（データの確認・準備・学習中）、進捗、残り時間、速度、学習曲線を表示します。
表示の■（停止）ボタンを押すと、表示だけが止まります。学習は裏で続きます。もう一度このセルを実行すれば表示が戻ります。

In [ ]:
mon.watch(CONFIG)

## 7. `.shk`（重みファイル）を書き出す

学習の途中の `checkpoint.pt` からでも書き出せます（保存は一時ファイルから改名するので、書き出し中に壊れることはありません）。
出力は `MyDrive/shikibu/runs/<CONFIG_NAME>/<CONFIG_NAME>.shk` です。書き出したあと、読み直して中身を確かめています。

In [ ]:
import subprocess

SHK = os.path.join(OUT_DIR, CONFIG_NAME + ".shk")
env = dict(os.environ, PYTHONPATH=os.path.join(REPO, "python"))
r = subprocess.run(["python", "-m", "shikibu.export",
                    "--checkpoint", os.path.join(OUT_DIR, "checkpoint.pt"), "--out", SHK],
                   cwd=REPO, env=env)
print("終了コード:", r.returncode)

## 8. 量子化の確認（`quantsim`）

エンジン（Nuenagon）で使う整数の量子化（§E4）を Python で再現し、float32 の重みとの差を測ります。
差が小さければ、エンジンに渡しても評価がほぼ変わりません。
表示される「z の差」「評価値の差（cp）」「損失」を控えてください。

注意：ここで使う PSV は学習に使ったファイルの先頭です。本当の確認には、学習に使わなかった局面（検証用）を使うのが望ましいです。

In [ ]:
first_psv = sorted(glob.glob(cfg["data"]["train_glob"]))[0]
r = subprocess.run(["python", "-m", "shikibu.quantsim", "--shk", SHK, "--psv", first_psv, "--n", "100000"],
                   cwd=REPO, env=env)
print("終了コード:", r.returncode)